In [1]:
import pandas as pd

In [ ]:
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data=pd.read_csv("samsum-train.csv")
test_data=pd.read_csv("samsum-test.csv")
val_data=pd.read_csv("samsum-validation.csv")
train_data.head()

In [ ]:
print(train_data.shape)
print(test_data.shape)

In [ ]:
# Data preprocessing

In [ ]:
import re

def clean_data(text):
  text=re.sub(r"\r\n"," ",text) #new lines
  text=re.sub(r"\s+"," ",text) #spaces
  text=re.sub(r"<.*?>"," ",text) #html
  text=text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].fillna("").astype(str).apply(clean_data)
test_data["dialogue"] = test_data["dialogue"].fillna("").astype(str).apply(clean_data)
val_data["dialogue"] = val_data["dialogue"].fillna("").astype(str).apply(clean_data)

In [ ]:
#Tokenization (text->numbers)

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [ ]:
#raw data=> tokenized input for fine tuning

def tokenize(data):
  input=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
  target=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

  input["labels"]=target["input_ids"]
  return input

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
#input_ids -> token's ids (1 means end of sequence,then 0 are padding)
#attention mask (1-valid tokenid, 0-padded tokenid)
#labels -> summary's ids (1 means end of sequence,then 0 are padding)
train_dataset[0]

## **Working with our model**

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import torch

if torch.backends.mps.is_available():
  device =torch.device("mps")
elif torch.cuda.is_available():
  device =torch.device("cuda")
else:
  device =torch.device("cpu")
print(device)
model.to(device)

In [ ]:
#Training Arguments

training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=6,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500 #(in 500 steps model will go from 0 to 5e05(defauls))
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
#train the model
trainer.train()

In [ ]:
model.save_pretrained("/content/drive/MyDrive/saved_summary_model")
tokenizer.save_pretrained("/content/drive/MyDrive/saved_summary_model")

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/saved_summary_model")
tokenizer=T5Tokenizer.from_pretrained("/content/drive/MyDrive/saved_summary_model")

In [ ]:
#Test the code logic for summarization
def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue)
  #tokenize
  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  )

  #generate summary(in terms of token ids)
  target=model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=input["attention_mask"],
      num_beams=4,
      early_stopping=True
  )

  #token id to text(decoding)
  summary=tokenizer.decode(target[0],skip_special_tokens=True)
  return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)